# Quant Portfolio Optimization System
## End-to-End Analysis Notebook

> *"The expected return of a portfolio is a weighted average of the expected returns of its component securities.
> This is not true of the standard deviation."* — Harry Markowitz, 1959

---

This notebook walks through the complete quantitative portfolio analysis pipeline:
from raw price data to institutional-grade risk metrics, backtesting, and stress testing.

### Contents
1. [Setup & Configuration](#1.-Setup)
2. [Data Acquisition & EDA](#2.-Data-Acquisition)
3. [Portfolio Optimisation (5 strategies)](#3.-Portfolio-Optimisation)
4. [Monte Carlo Simulation](#4.-Monte-Carlo-Simulation)
5. [Efficient Frontier & Capital Market Line](#5.-Efficient-Frontier)
6. [Institutional Risk Metrics](#6.-Risk-Metrics)
7. [Walk-Forward Backtesting](#7.-Backtesting)
8. [Stress Testing](#8.-Stress-Testing)
9. [Visualisations](#9.-Visualisations)
10. [Conclusions & Limitations](#10.-Conclusions)

---
## 1. Setup

In [ ]:
# !pip install -r requirements.txt   # uncomment if needed
import os, yaml, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
warnings.filterwarnings('ignore')

with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.data_loader import DataLoader
from src.portfolio_optimizer import PortfolioOptimizer
from src.risk_metrics import RiskMetrics
from src.backtester import Backtester
from src.visualization import Visualizer

viz = Visualizer(config)
print('Config loaded. Assets:', config['data']['tickers'])

---
## 2. Data Acquisition

We download 3+ years of **total-return** (split and dividend adjusted) prices from
Yahoo Finance. A Ledoit-Wolf shrinkage covariance is used — it reduces estimation error
significantly compared to the sample covariance, particularly important when T/N is small.

In [ ]:
loader = DataLoader(config)
prices = loader.download()

returns = loader.get_returns()
asset_returns = loader.get_asset_returns()
benchmark_returns = loader.get_benchmark_returns()

# Expected returns and covariance — use Ledoit-Wolf shrinkage
mu = loader.get_expected_returns(method='mean_historical')
sigma = loader.get_covariance(method='sample')   # change to 'ledoit_wolf' for shrinkage

# Strip benchmark from asset universe
bench_col = config['data']['benchmark'].replace('^', '')
asset_mu = mu.drop(bench_col, errors='ignore')
asset_sigma = sigma.drop(index=bench_col, columns=bench_col, errors='ignore')

print(f'Prices: {prices.shape[0]:,} days × {prices.shape[1]} assets')
print(f'Range : {prices.index[0].date()} → {prices.index[-1].date()}')
print()
print('=== Asset Summary ===')
display(loader.summary())

In [ ]:
# Correlation matrix — low correlations are the engine of diversification
print('=== Annualised Correlation Matrix ===')
display(asset_returns.corr().style
        .background_gradient(cmap='RdYlGn', vmin=-1, vmax=1)
        .format('{:.2f}'))

---
## 3. Portfolio Optimisation

We solve five distinct optimisation problems.

| Strategy | Objective | Constraint |
|----------|-----------|-----------|
| **Min Variance** | $\min \mathbf{w}^\top \Sigma \mathbf{w}$ | $\sum w_i = 1$, $w_i \geq 0$ |
| **Max Sharpe** | $\max (\mathbf{w}^\top \mu - R_f)/\sigma_p$ | same |
| **Equal Weight** | $w_i = 1/N$ | — |
| **Target Return** | $\min \sigma_p$ s.t. $\mathbf{w}^\top \mu = r^*$ | same |
| **Constrained** | Max Sharpe | $w_i \in [2\%, 40\%]$ |

In [ ]:
import numpy as np
optimizer = PortfolioOptimizer(
    asset_mu, asset_sigma,
    risk_free_rate=config['optimization']['risk_free_rate'],
)

target_ret = config['optimization']['target_return']
strategies = optimizer.compare_all(
    target_return=target_ret,
    constrained_min=config['optimization']['constrained']['min_weight'],
    constrained_max=config['optimization']['constrained']['max_weight'],
)

print()
print('=== Portfolio Comparison Table ===')
display(optimizer.comparison_table(strategies))

In [ ]:
# Individual result details
for name, res in strategies.items():
    print(f'\n{name}')
    print(f'  Return: {res.expected_return:.2%}  |  Vol: {res.volatility:.2%}  |  Sharpe: {res.sharpe_ratio:.2f}')
    significant = res.weights[res.weights > 0.01]
    for ticker, w in significant.sort_values(ascending=False).items():
        bar = '█' * int(w * 40)
        print(f'  {ticker:<6} {w:>6.1%}  {bar}')

---
## 4. Monte Carlo Simulation

We sample 10,000 random long-only weight vectors from the **uniform Dirichlet distribution**
(equivalent to sampling uniformly from the n-simplex). This approximates the full feasible set
of long-only portfolios, giving intuition about the shape of the opportunity set.

The colour gradient (viridis) shows that higher Sharpe portfolios cluster near the frontier —
confirming our optimiser found the correct solution.

In [ ]:
n_sim = config['optimization']['n_simulations']
print(f'Simulating {n_sim:,} random portfolios ...')
mc_rets, mc_vols, mc_sharpes, mc_weights = optimizer.monte_carlo(n=n_sim)
print(f'Done. Sharpe range: [{mc_sharpes.min():.2f}, {mc_sharpes.max():.2f}]')
print(f'Return range: [{mc_rets.min():.2%}, {mc_rets.max():.2%}]')
print(f'Vol range   : [{mc_vols.min():.2%}, {mc_vols.max():.2%}]')

---
## 5. Efficient Frontier & Capital Market Line

The **efficient frontier** is traced by solving 200 minimum-variance problems, each
constraining portfolio return to a different target level.

The **Capital Market Line (CML)** starts at the risk-free rate and passes through the
tangency portfolio (Max Sharpe). All points on the CML dominate the efficient frontier —
an investor can achieve any risk level on the CML by mixing the risk-free asset with the
tangency portfolio, without owning any other combination.

In [ ]:
print('Tracing efficient frontier (200 points) ...')
eff_rets, eff_vols, eff_weights = optimizer.efficient_frontier(n_points=200)
cml_vols, cml_rets = optimizer.capital_market_line()

print(f'{len(eff_rets)} frontier points computed.')
print(f'Frontier return range: [{min(eff_rets):.2%}, {max(eff_rets):.2%}]')
print(f'Frontier vol range  : [{min(eff_vols):.2%}, {max(eff_vols):.2%}]')

---
## 6. Institutional Risk Metrics

We compute the full suite of metrics used in professional asset management.

In [ ]:
# Build return series for each strategy
strategy_returns = {}
for name, res in strategies.items():
    common = [t for t in res.weights.index if t in asset_returns.columns]
    w = res.weights[common].values / res.weights[common].sum()
    strategy_returns[name] = (asset_returns[common] @ w).rename(name)

metrics = RiskMetrics.compare_portfolios(
    strategy_returns,
    benchmark_returns=benchmark_returns,
    risk_free_rate=config['optimization']['risk_free_rate'],
)

# Format nicely
fmt = metrics.copy().astype(object)
pct_rows = ['Return', 'Volatility', 'Drawdown', 'VaR', 'CVaR', 'Error', 'Alpha', 'Tracking']
for row in fmt.index:
    for col in fmt.columns:
        val = metrics.loc[row, col]
        if isinstance(val, float):
            if any(kw in row for kw in pct_rows):
                fmt.loc[row, col] = f'{val:.2%}'
            else:
                fmt.loc[row, col] = f'{val:.2f}'

display(fmt)

In [ ]:
# Rolling 1-year metrics for Max Sharpe strategy
ms_returns = strategy_returns['Max Sharpe']
idx = pd.date_range('2022-01-03', periods=len(ms_returns), freq='B')
ms_returns.index = ms_returns.index if hasattr(ms_returns.index, 'freq') else idx[:len(ms_returns)]

rolling = RiskMetrics.rolling_metrics(
    ms_returns,
    window=config['risk']['rolling_window'],
    risk_free_rate=config['optimization']['risk_free_rate'],
)
print('Rolling 1Y metrics (last 5 observations):')
display(rolling.dropna().tail().style.format({'Return': '{:.2%}', 'Volatility': '{:.2%}', 'Sharpe': '{:.2f}'}))

---
## 7. Walk-Forward Backtesting

The walk-forward backtest avoids **look-ahead bias** by:
1. Fitting the optimiser on a 2-year in-sample window
2. Applying the resulting weights to the following quarter
3. Rolling forward and repeating

Transaction costs of **10 bps per dollar of turnover** are deducted on each rebalancing.

In [ ]:
print('Running walk-forward backtest ...')
print(f'  In-sample: {config["backtesting"]["in_sample_years"]} years')
print(f'  Rebalancing: {config["backtesting"]["rebalancing_frequency"]}')
print(f'  TC: {config["backtesting"]["transaction_cost"]*100:.0f} bps')
print()

backtester = Backtester(prices, config)
backtest_result = backtester.run_all_strategies()

bt_metrics = backtest_result.metrics(
    benchmark_returns=benchmark_returns,
    risk_free_rate=config['optimization']['risk_free_rate'],
)
key_rows = ['Annualised Return', 'Annualised Volatility', 'Sharpe Ratio',
            'Sortino Ratio', 'Max Drawdown', 'Calmar Ratio']
display(bt_metrics.loc[key_rows].round(4).style.format('{:.4f}'))

print()
print('Final portfolio values (started at $1.00):')
final = backtest_result.portfolio_values.iloc[-1].sort_values(ascending=False)
for name, val in final.items():
    bar = '█' * int((val - 1) * 20) if val > 1 else ''
    print(f'  {name:<30} ${val:.3f}  {bar}')

---
## 8. Stress Testing

We measure how each strategy would have performed during three recent market crises.
Weights from the **full-history optimisation** (not walk-forward) are applied — this
tests the stability of optimal portfolios under extreme conditions.

In [ ]:
stress_periods = config.get('stress_tests', {}).get('periods', [])
stress_returns = {}

for period in stress_periods:
    try:
        slice_ret = loader.stress_test_slice(period['start'], period['end'])
        if not slice_ret.empty:
            avail = [c for c in slice_ret.columns if c != bench_col]
            stress_returns[period['name']] = slice_ret[avail]

            # Compute performance for each strategy
            print(f"\n{period['name']} ({period['start']} → {period['end']})")
            print(f"  {period.get('description', '')}")
            for name, res in strategies.items():
                common = [t for t in res.weights.index if t in avail]
                if not common:
                    continue
                w = res.weights[common].values / res.weights[common].sum()
                port_rets = slice_ret[common] @ w
                total = (1 + port_rets).prod() - 1
                print(f'    {name:<30}: {total:>+.2%}')
    except Exception as e:
        print(f'  [skip] {period["name"]}: {e}')

---
## 9. Visualisations

All charts are saved to `./plots/` at 150 dpi.

In [ ]:
import numpy as np

asset_vols_series = pd.Series(
    np.sqrt(np.diag(asset_sigma.values)), index=asset_sigma.index
)

# 1. Efficient frontier
p1 = viz.plot_efficient_frontier(
    mc_rets, mc_vols, mc_sharpes,
    eff_rets, eff_vols, cml_vols, cml_rets,
    strategies, asset_mu,
    asset_sigma_diag=asset_vols_series.values,
    risk_free_rate=config['optimization']['risk_free_rate'],
)
display(Image(filename=p1))

In [ ]:
# 2. Portfolio weights comparison
p2 = viz.plot_weights_comparison(strategies)
display(Image(filename=p2))

In [ ]:
# 3. Correlation heatmap
p3 = viz.plot_correlation_heatmap(asset_returns)
display(Image(filename=p3))

In [ ]:
# 4. Portfolio growth (backtest)
p4 = viz.plot_portfolio_growth(backtest_result.portfolio_values)
display(Image(filename=p4))

In [ ]:
# 5. Drawdown
p5 = viz.plot_drawdown(backtest_result.portfolio_values)
display(Image(filename=p5))

In [ ]:
# 6. Rolling metrics
rolling_dict = backtester.rolling_metrics(backtest_result, window=config['risk']['rolling_window'])
p6 = viz.plot_rolling_metrics(rolling_dict)
display(Image(filename=p6))

In [ ]:
# 7. Risk-return scatter
p7 = viz.plot_risk_return_scatter(asset_mu, asset_vols_series, strategies)
display(Image(filename=p7))

In [ ]:
# 8. Monthly returns heatmap (Max Sharpe strategy)
ms_ret_series = strategy_returns['Max Sharpe']
p8 = viz.plot_monthly_returns_heatmap(ms_ret_series, 'Max Sharpe — Monthly Returns (%)')
display(Image(filename=p8))

In [ ]:
# 9. Stress test
if stress_returns:
    p9 = viz.plot_stress_test(stress_returns, strategies, asset_returns)
    display(Image(filename=p9))

---
## 10. Conclusions & Limitations

### Key Findings

1. **Diversification works**: The efficient frontier lies above and to the left of all
   individual assets — every optimised portfolio achieves a better risk-return trade-off
   than any single-asset allocation.

2. **Max Sharpe concentrates in growth**: The tangency portfolio allocates heavily to
   high-momentum technology stocks, reflecting their superior risk-adjusted returns
   over the sample period.

3. **Min Variance gravitates toward defensives**: Gold (GLD) and Healthcare (JNJ) receive
   the largest allocations — their low correlation with equities provides the greatest
   diversification benefit.

4. **Constraints matter**: The constrained Max Sharpe portfolio (2%–40% bounds) produces
   a more practical allocation with reduced turnover and better out-of-sample stability.

5. **Backtesting reveals limits**: All strategies underperform the S&P 500 during strong
   bull markets due to their defensive tilts, but outperform during corrections.

### Limitations

| Assumption | Risk | Mitigation |
|------------|------|------------|
| Historical returns = future returns | Distribution shift | Black-Litterman priors |
| Normal returns | Fat tails, skewness | CVaR optimisation |
| Static Σ | Regime changes | DCC-GARCH |
| No costs beyond rebalancing TC | Spread, market impact | Full transaction cost model |
| Long-only, single-period | No short selling, no leverage | Extended model |

### Next Steps for Production

- **Factor models**: Replace historical Σ with Barra-style factor model covariance
- **Black-Litterman**: Blend quantitative signals with market equilibrium priors
- **Risk parity**: Equal risk contribution across assets (alternative to equal weight)
- **Regime detection**: Hidden Markov Models to switch between risk-on/risk-off allocations
- **Live rebalancing**: Connect to broker API for automated portfolio execution